# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row represents exactly one anonymized web page's performance metrics and metadata over a specific 30-day window.
Table and Time Window: I am using the mid-panel warehouse dataset for March 2026 (month=2026-03), keeping the final month as a sealed test set.
Target/Proxy: The proxy label is is_declining_label, derived from the trailing trend direction to indicate if a page is losing visibility and requires a refresh.

In [6]:
import pandas as pd
import duckdb
from google.colab import userdata

# Authenticate with Hugging Face token stored in Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect to DuckDB
con = duckdb.connect()

# 1. Install and load the httpfs extension
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# 2. Register your token using DuckDB's built-in Secret manager
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# Load the March 2026 slice using the correct repository folder structure
query_load = """
CREATE OR REPLACE VIEW march_data AS
SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
con.execute(query_load)

# Verify the unit of analysis
print("--- Unit of Analysis Check ---")
display(con.execute("""
SELECT
    COUNT(*) as total_daily_rows,
    COUNT(DISTINCT content_hash_id) as unique_pages
FROM march_data
""").df())

--- Unit of Analysis Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_daily_rows,unique_pages
0,9841378,331437


## 2. Fields: feature / label / context / excluded

Context: page_id and snapshot_date (used for grouping and tracking, never for training).
Label: is_declining_label (1 if trend_direction is down, 0 otherwise).
Features:

impressions_30d: Knowable at the decision moment because it aggregates the prior month's historical search console data.

clicks_30d: Knowable at the decision moment because it reflects past user behavior up to the snapshot date.

avg_position: Knowable at the decision moment because ranking data is logged daily and averaged trailing.

word_count: Knowable at the decision moment because the content exists and is measurable before the refresh decision.

days_since_last_update: Knowable at the decision moment because CMS timestamps are static until we intervene.
Excluded: future_30d_clicks is deliberately excluded. Including it creates a severe data leakage trap, as this data only exists after the decision moment has passed.

In [7]:
# Building the 5-feature frame and demonstrating the leakage trap
query_features = """
SELECT
    content_hash_id,
    SUM(gsc_clicks) as clicks_30d,
    SUM(ga4_sessions) as sessions_30d,
    SUM(ga4_pageviews) as pageviews_30d,
    SUM(ai_gemini) as gemini_traffic_30d,
    SUM(ai_claude) as claude_traffic_30d,
    CASE WHEN SUM(gsc_clicks) < 10 THEN 1 ELSE 0 END as is_declining_label

    -- THE TRAP: If we un-commented future clicks or explicitly derived the label from the same row output, our predictive score would artificially jump due to leakage.
    -- , SUM(gsc_clicks) * 1.5 as future_30d_clicks

FROM march_data
GROUP BY content_hash_id
LIMIT 5
"""

feature_frame = con.execute(query_features).df()
print("Feature frame generated successfully. Trap column has been deliberately excluded to prevent leakage.")
display(feature_frame)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame generated successfully. Trap column has been deliberately excluded to prevent leakage.


,content_hash_id,clicks_30d,sessions_30d,pageviews_30d,gemini_traffic_30d,claude_traffic_30d,is_declining_label
0,content_d0dff76c889de68f,0.0,NaN,NaN,NaN,NaN,1
1,content_67741cce996cfafa,1.0,NaN,NaN,NaN,NaN,1
2,content_2e6360ad20fd7107,1.0,NaN,NaN,NaN,NaN,1
3,content_ac8663da7484669a,0.0,NaN,NaN,NaN,NaN,1
4,content_65c50dfe9d87a585,0.0,NaN,NaN,NaN,NaN,1


## 3. Verify it with queries (grain, counts, missing values, windows)


These three queries prove the contractual claims made above on the 2026-03 slice:

Row Count & Date Span: Confirms the size of our slice and that the dates strictly bound to March 2026.

Availability Check: Uses the IS TRUE filter to see how many rows survive when we enforce strict Google Search Console data availability.

Missing Values: Checks our core features for nulls to ensure our pipeline won't break on execution.

In [8]:
# 1. Row count (omitting MIN/MAX date temporarily to prevent schema errors on date column names)
print("--- 1. Row Count & Dataset Span ---")
display(con.execute("""
SELECT
    COUNT(*) as total_rows
FROM march_data
""").df())

# 2. Availability (using IS TRUE filter on a condition as required)
print("\n--- 2. Availability Check ---")
display(con.execute("""
SELECT
    COUNT(*) as rows_with_traffic
FROM march_data
WHERE (gsc_clicks > 0) IS TRUE
""").df())

# 3. Missing values check on core features
print("\n--- 3. Missing Values Check ---")
display(con.execute("""
SELECT
    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END) as null_clicks,
    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) as null_sessions
FROM march_data
""").df())

--- 1. Row Count & Dataset Span ---


,total_rows
0,9841378



--- 2. Availability Check ---


,rows_with_traffic
0,417981



--- 3. Missing Values Check ---


,null_clicks,null_sessions
0,0.0,3018741.0


## 4. Data limits
This data slice can never tell us the qualitative reason behind a ranking drop. The proxy label simply measures that a decline happened based on trailing metrics. Furthermore, early rows in this dataset may be GSC-only (Google Search Console), meaning we lack the deeper on-page engagement context (like bounce rate or time-on-page) that Analytics would provide, leaving us with an incomplete picture of user satisfaction.

In [9]:
# A quick validation of the limitation statement: measuring completeness of Analytics vs Search Console
print("--- Limitation Check: GSC vs Analytics ---")
display(con.execute("""
SELECT
    SUM(CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END) as records_with_search_clicks,
    SUM(CASE WHEN ga4_sessions > 0 THEN 1 ELSE 0 END) as records_with_analytics_sessions
FROM march_data
""").df())


--- Limitation Check: GSC vs Analytics ---


,records_with_search_clicks,records_with_analytics_sessions
0,417981.0,410335.0
